In [1]:
# 10_mitigation — Cell 1: cue-masking + TF-IDF/LR re-eval on domain-held-out splits, c2020 & c2025
import os, re, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
SEED = 0

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}

URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
# common Bangla news boilerplate phrases — extend this list as needed after inspecting outputs
BOILERPLATE = [
    r"সৌজন্যে[:：]?\s*\S+", r"সূত্র[:：]?\s*\S+", r"রিপোর্ট[:：]?\s*\S+",
    r"তারিখ\s*[:：]?\s*[\d/.\-]+", r"প্রকাশ\s*[:：]?\s*[\d/.\-]+",
    r"\b\d{1,2}[/.\-]\d{1,2}[/.\-]\d{2,4}\b",
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE))

def mask_cues(text, domain):
    t = str(text)
    t = URL_RE.sub(" [URL] ", t)
    t = EMAIL_RE.sub(" [EMAIL] ", t)
    t = BOILERPLATE_RE.sub(" [META] ", t)
    if isinstance(domain, str) and domain.strip():
        # strip the raw domain/site name string itself if it leaks into the body/byline
        dom_stub = re.escape(domain.strip().split(".")[0])
        if len(dom_stub) > 2:
            t = re.sub(dom_stub, "[SITE]", t, flags=re.IGNORECASE)
    return t

for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:1500]
    df["domain"] = df["domain"].fillna("unk")
    df["text_masked"] = [mask_cues(t, d) for t, d in zip(df["text"], df["domain"])]

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

def fit_eval(tr, te, text_col, tag):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(v.fit_transform(tr[text_col]), tr["label"])
    s = clf.decision_function(v.transform(te[text_col]))
    p = (s > 0).astype(int)
    return dict(split=tag, n_train=len(tr), n_test=len(te), **ev(te["label"], p, s))

rows = []
for name, df in CORP.items():
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    tr_idx, te_idx = next(gss.split(df, groups=df["domain"]))
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]

    rows.append(dict(corpus=name, variant="raw_text", **fit_eval(tr, te, "text", "domain_heldout")))
    rows.append(dict(corpus=name, variant="cue_masked", **fit_eval(tr, te, "text_masked", "domain_heldout")))

    # quick eyeball: how much text actually changed?
    changed = (df["text"] != df["text_masked"]).mean()
    print(f"[{name}] fraction of rows altered by masking: {changed:.3f}")

res = pd.DataFrame(rows)
res.to_csv(f"{R}/cue_masking.csv", index=False)
print(res.to_string(index=False))

[c2020] fraction of rows altered by masking: 0.146
[c2025] fraction of rows altered by masking: 0.102
corpus    variant          split  n_train  n_test  bacc    f1   auc
 c2020   raw_text domain_heldout    34303   15518 0.768 0.812 0.969
 c2020 cue_masked domain_heldout    34303   15518 0.770 0.814 0.970
 c2025   raw_text domain_heldout     3280     699 0.589 0.535 0.688
 c2025 cue_masked domain_heldout     3280     699 0.589 0.535 0.692
